# SaaS Product Support — Multi-Agent Workflow

This notebook implements a goal-oriented multi-agent workflow for a SaaS product support use case (technical issue diagnosis). It includes agent definitions, handoff formats, escalation logic, and sample runs demonstrating end-to-end behavior.


## Architecture Explanation

**Agents (4):**
- **TechnicalDiagnosisAgent**: Parse user symptom, identify probable root cause, confidence score, and quick checks to run.
- **ProductKnowledgeAgent**: Apply non-RAG product rules and policies (config limits, known behaviours) to validate diagnosis and suggest safe mitigations.
- **TroubleshootingResponseAgent**: Produce step-by-step user-facing troubleshooting instructions, triage severity, and estimate resolution probability.
- **EngineeringEscalationAgent**: Decide escalation when risk/uncertainty/edge-case detected; format the engineering ticket payload.

**Handoff format**: Each agent receives a dictionary like:

```
{"user_query": str, "context": {...}, "diagnosis": {...}}
```

Agents augment the dict with `diagnosis`, `confidence`, `actions`, and `escalation_reason`.

**Escalation rules (examples):**
- If combined confidence < 0.5 -> escalate.
- If issue maps to data-loss, security, or production outage -> escalate immediately.
- If troubleshooting steps include code-level fixes or schema migrations -> escalate.

**Design rationale**: Keep agents narrowly-scoped so each step's output is structured and checkable. Escalation is explicit and explainable (reasons + evidence).


## Sample inputs & expected behavior

- A clear `auth` issue with high-confidence should return step-by-step auth checks and typically not escalate unless product rules indicate outage.
- An `api` issue with server error (`500`) reduces confidence and triggers escalation with logs collected.
- Ambiguous queries produce low confidence and a structured escalation payload for engineers (reasons + evidence).


## Next steps

- Run this notebook to see the simulated flows.
- Optionally, connect the agents to real LLM calls or integrate with CrewAI for autonomous orchestration.
- Expand product rules and escalate payload fields to match your team's ticketing system.


## CrewAI Integration

The notebook now includes **full CrewAI orchestration** of the 4 agents:

1. **Technical Diagnosis Specialist** — Parse symptoms & assign confidence
2. **Product Knowledge Specialist** — Apply product rules/policies
3. **Troubleshooting Specialist** — Generate user-facing steps & severity
4. **Engineering Escalation Manager** — Decide escalation & format tickets

### How to use CrewAI:

```bash
# Install CrewAI
pip install crewai

# Set valid OpenAI API key
$env:OPENAI_API_KEY = "your-key-here"
```

Then run the notebook. `run_pipeline_auto(query, use_crewai=True)` will automatically use CrewAI if available.

**Pipeline selection order:**
1. CrewAI (if `use_crewai=True` and CrewAI installed)
2. OpenAI LLM (if `OPENAI_API_KEY` set and OpenAI installed)
3. Deterministic fallback (always available, no dependencies)


## Expected Output

When running the pipeline with deterministic agents, you should see:

```
CrewAI not installed; run `pip install crewai` to enable CrewAI orchestration
Running with deterministic agents (use_crewai=False):
```

Followed by diagnostic results for each example query showing diagnosis, confidence, response steps, and escalation decisions.


In [ ]:
# Deterministic / LLM / CrewAI agent implementations
# This cell defines deterministic fallback agents, LLM wrappers, and full CrewAI orchestration.

import os
import json
import re
import asyncio
from typing import Dict, Any, List

# Deterministic agent implementations


def DeterministicTechnicalDiagnosisAgent(user_query: str, context: Dict[str, Any]) -> Dict[str, Any]:
    q = user_query.lower()
    diagnosis = {}
    if 'login' in q or 'sign in' in q or 'authenticate' in q:
        diagnosis['issue_type'] = 'auth'
        diagnosis['probable_cause'] = 'invalid_credentials_or_sso'
        confidence = 0.8
        actions = ['check-credentials', 'check-sso-status']
    elif 'api' in q or 'endpoint' in q or '500' in q or 'error' in q:
        diagnosis['issue_type'] = 'api'
        diagnosis['probable_cause'] = 'bad_request_or_backend_error'
        confidence = 0.6
        actions = ['collect-logs', 'reproduce-call']
    elif 'feature' in q or 'how to' in q or 'usage' in q:
        diagnosis['issue_type'] = 'usage'
        diagnosis['probable_cause'] = 'user-error_or-missing-docs'
        confidence = 0.7
        actions = ['show-docs', 'example-commands']
    else:
        diagnosis['issue_type'] = 'unknown'
        diagnosis['probable_cause'] = 'needs-investigation'
        confidence = 0.4
        actions = ['collect-more-info']
    return {'diagnosis': diagnosis, 'confidence': confidence, 'actions': actions, 'evidence': q}


def DeterministicProductKnowledgeAgent(state: Dict[str, Any], product_rules: Dict[str, Any]) -> Dict[str, Any]:
    diag = state['diagnosis']
    issue = diag['issue_type']
    augmentation = {}
    if issue == 'auth':
        if product_rules.get('sso_outage', False):
            augmentation['note'] = 'Known SSO outage'
            state['confidence'] = min(state['confidence'] * 0.9, 0.95)
            augmentation['escalate'] = True
        else:
            augmentation['note'] = 'Follow normal auth troubleshooting'
    elif issue == 'api':
        if '500' in state.get('evidence', ''):
            augmentation['note'] = 'Server error observed; collect logs'
            augmentation['escalate'] = True
        else:
            augmentation['note'] = 'Apply API troubleshooting rules'
    else:
        augmentation['note'] = 'No special product rule'
    state.setdefault('notes', []).append(augmentation)
    return state


def DeterministicTroubleshootingResponseAgent(state: Dict[str, Any]) -> Dict[str, Any]:
    diag = state['diagnosis']
    issue = diag['issue_type']
    response = {}
    if issue == 'auth':
        response['steps'] = [
            'Confirm username/email is correct',
            'Reset password or try SSO login flow',
            'If using SSO, check IdP status'
        ]
        response['severity'] = 'medium'
    elif issue == 'api':
        response['steps'] = [
            'Collect request/response headers and payload',
            'Check service status and recent deploys',
            'Retry the request and capture logs'
        ]
        response['severity'] = 'high' if state['confidence'] < 0.6 else 'medium'
    elif issue == 'usage':
        response['steps'] = ['Provide docs link', 'Give example usage snippet']
        response['severity'] = 'low'
    else:
        response['steps'] = ['Ask user for more context: screenshots, logs, steps to reproduce']
        response['severity'] = 'unknown'
    state['response'] = response
    return state


def DeterministicEngineeringEscalationAgent(state: Dict[str, Any]) -> Dict[str, Any]:
    escalate = False
    reasons: List[str] = []
    if state.get('confidence', 1.0) < 0.5:
        escalate = True
        reasons.append('low_confidence')
    resp = state.get('response', {})
    if resp.get('severity') == 'high':
        escalate = True
        reasons.append('high_severity')
    for note in state.get('notes', []):
        if note.get('escalate'):
            escalate = True
            reasons.append('product_rule')
    if escalate:
        ticket = {
            'title': f"Escalation: {state['diagnosis']['issue_type']}",
            'description': state.get('evidence', '') + '\n' + str(state.get('diagnosis', {})),
            'attachments': state.get('logs', [])
        }
        state['escalation'] = {'escalate': True, 'reasons': reasons, 'ticket': ticket}
    else:
        state['escalation'] = {'escalate': False, 'reasons': reasons}
    return state


def run_pipeline(user_query: str, product_rules: Dict[str, Any] = {}) -> Dict[str, Any]:
    s1 = DeterministicTechnicalDiagnosisAgent(user_query, {})
    state = {
        'user_query': user_query,
        'diagnosis': s1['diagnosis'],
        'confidence': s1['confidence'],
        'actions': s1['actions'],
        'evidence': s1['evidence']
    }
    state = DeterministicProductKnowledgeAgent(state, product_rules)
    state = DeterministicTroubleshootingResponseAgent(state)
    state = DeterministicEngineeringEscalationAgent(state)
    return state


def is_valid_pipeline_result(result: Any) -> bool:
    if not isinstance(result, dict):
        return False
    if not isinstance(result.get('diagnosis'), dict):
        return False
    if 'confidence' not in result:
        return False
    if not isinstance(result.get('response'), dict):
        return False
    if not isinstance(result.get('escalation'), dict):
        return False
    return True


def _coerce_diagnosis(payload: Any) -> Dict[str, Any]:
    if isinstance(payload, dict):
        if isinstance(payload.get('diagnosis'), dict):
            diag = payload['diagnosis']
        else:
            diag = payload
        issue_type = diag.get('issue_type') or payload.get('issue_type') or 'unknown'
        probable_cause = diag.get('probable_cause') or payload.get('probable_cause') or 'needs-investigation'
        return {'issue_type': issue_type, 'probable_cause': probable_cause}
    return {'issue_type': 'unknown', 'probable_cause': 'needs-investigation'}


def _coerce_response(payload: Any) -> Dict[str, Any]:
    response = payload.get('response') if isinstance(payload, dict) and isinstance(payload.get('response'), dict) else payload
    if isinstance(response, dict):
        steps = response.get('steps') or ['Ask user for more context: screenshots, logs, steps to reproduce']
        severity = response.get('severity') or 'unknown'
        return {'steps': steps, 'severity': severity}
    return {'steps': ['Ask user for more context: screenshots, logs, steps to reproduce'], 'severity': 'unknown'}


def _normalize_state(state: Dict[str, Any], user_query: str) -> Dict[str, Any]:
    if not isinstance(state, dict):
        state = {}
    state.setdefault('user_query', user_query)
    state.setdefault('diagnosis', {'issue_type': 'unknown', 'probable_cause': 'needs-investigation'})
    state.setdefault('confidence', 0.4)
    state.setdefault('actions', ['collect-more-info'])
    state.setdefault('evidence', user_query)
    state.setdefault('response', {'steps': ['Ask user for more context: screenshots, logs, steps to reproduce'], 'severity': 'unknown'})
    state.setdefault('escalation', {'escalate': False, 'reasons': ['default_fallback']})
    state.setdefault('notes', [])
    return state


# ======================== LLM Integration (OpenAI backend) ========================
DEFAULT_LLM_MODEL = 'gpt-3.5-turbo'
USE_LLM = bool(os.getenv('OPENAI_API_KEY'))

openai_client = None
if USE_LLM:
    try:
        from openai import OpenAI
        openai_client = OpenAI()  # Automatically reads OPENAI_API_KEY from environment
        print(f'✓ OpenAI client initialized (using {DEFAULT_LLM_MODEL})')
    except Exception as e:
        print(f'✗ openai package not available; install with `pip install openai` to enable LLM agents. Error: {e}')
        USE_LLM = False


def call_llm_json(prompt: str, model: str = DEFAULT_LLM_MODEL) -> dict:
    if not USE_LLM or openai_client is None:
        raise RuntimeError('LLM not enabled (missing OPENAI_API_KEY or openai package)')
    messages = [
        {"role": "system", "content": "You are a precise assistant that outputs only valid JSON matching the requested schema."},
        {"role": "user", "content": prompt}
    ]
    resp = openai_client.chat.completions.create(model=model, messages=messages, temperature=0)
    text = resp.choices[0].message.content
    text = text.strip()
    if text.startswith('```') and text.endswith('```'):
        parts = text.split('\n')
        text = '\n'.join(parts[1:-1])
    try:
        return json.loads(text)
    except Exception:
        m = re.search(r'\{.*\}', text, re.S)
        if m:
            return json.loads(m.group(0))
        raise


def TechnicalDiagnosisAgent(user_query: str, context: dict) -> dict:
    prompt = f"""
    Produce a JSON object with keys: diagnosis (object with issue_type and probable_cause), confidence (0-1 float), actions (array of strings), evidence (original user query).
    User query: {user_query}
    """
    return call_llm_json(prompt)


def ProductKnowledgeAgent(state: dict, product_rules: dict) -> dict:
    prompt = f"""
    Given the following state and product rules, return the updated state as JSON. Keys to include/augment: notes (array), confidence (may be adjusted), escalate flags inside notes when appropriate.
    State: {json.dumps(state)}
    Product rules: {json.dumps(product_rules)}
    """
    return call_llm_json(prompt)


def TroubleshootingResponseAgent(state: dict) -> dict:
    prompt = f"""
    Given the state, return a JSON object with a response key containing steps (array of step strings) and severity (one of low, medium, high, unknown).
    State: {json.dumps(state)}
    """
    return call_llm_json(prompt)


def EngineeringEscalationAgent(state: dict) -> dict:
    prompt = f"""
    Decide whether to escalate. Return JSON: {{"escalation": {{"escalate": bool, "reasons": [str], "ticket": {{...}}}}}}. Include evidence and diagnosis in ticket.description if escalating.
    State: {json.dumps(state)}
    """
    return call_llm_json(prompt)


def run_pipeline_llm(user_query: str, product_rules: dict = {}) -> dict:
    if not USE_LLM:
        raise RuntimeError('LLM not enabled; set OPENAI_API_KEY and install openai')
    s1 = TechnicalDiagnosisAgent(user_query, {})
    state = {
        'user_query': user_query,
        'diagnosis': _coerce_diagnosis(s1),
        'confidence': s1.get('confidence', 0.4) if isinstance(s1, dict) else 0.4,
        'actions': s1.get('actions', ['collect-more-info']) if isinstance(s1, dict) and isinstance(s1.get('actions'), list) else ['collect-more-info'],
        'evidence': s1.get('evidence', user_query) if isinstance(s1, dict) else user_query
    }
    state = _normalize_state(state, user_query)
    if isinstance(s1, dict):
        if isinstance(s1.get('diagnosis'), dict):
            state['diagnosis'] = s1['diagnosis']
        elif isinstance(s1.get('issue_type'), str):
            state['diagnosis'] = {'issue_type': s1['issue_type'], 'probable_cause': s1.get('probable_cause', 'needs-investigation')}
    product_state = ProductKnowledgeAgent(state, product_rules)
    if isinstance(product_state, dict):
        state.update(product_state)
    state = _normalize_state(state, user_query)
    response_state = TroubleshootingResponseAgent(state)
    if isinstance(response_state, dict):
        state.update(response_state)
    state['response'] = _coerce_response(state)
    escalation_state = EngineeringEscalationAgent(state)
    if isinstance(escalation_state, dict):
        state.update(escalation_state)
    if not isinstance(state.get('escalation'), dict):
        state['escalation'] = {'escalate': False, 'reasons': ['default_fallback']}
    return _normalize_state(state, user_query)

# ======================== CrewAI Integration ========================
USE_CREWAI = False
try:
    from crewai import Agent, Task, Crew
    USE_CREWAI = True
    print('✓ CrewAI installed')
except ImportError:
    print('✗ CrewAI not installed; run `pip install crewai` to enable CrewAI orchestration')


def _is_running_in_event_loop() -> bool:
    try:
        asyncio.get_running_loop()
        return True
    except RuntimeError:
        return False


def can_use_crewai(requested: bool) -> bool:
    """Only enable CrewAI when the user explicitly requests it and all prerequisites are available."""
    if not requested:
        return False
    if not USE_CREWAI:
        return False
    if _is_running_in_event_loop():
        print('⚠ Detected an active event loop; skipping CrewAI in this notebook runtime and using LLM/deterministic fallback instead.')
        return False
    if not USE_LLM or openai_client is None:
        return False
    if not os.getenv('OPENAI_API_KEY'):
        return False
    return True


if USE_CREWAI and USE_LLM:
    # Define CrewAI agents
    tech_diagnosis_agent = Agent(
        role="Technical Diagnosis Specialist",
        goal="Parse user symptoms, identify the probable root cause, assign confidence scores, and recommend diagnostic checks.",
        backstory="Expert at analyzing support tickets for SaaS products. Recognizes auth, API, integration, and data issues.",
        llm=None
    )
    
    product_knowledge_agent = Agent(
        role="Product Knowledge Specialist",
        goal="Apply product rules and policies to validate diagnosis and suggest safe mitigations without requiring code-level fixes.",
        backstory="Knows product-specific configurations, safety limits, service status, and escalation policies.",
        llm=None
    )
    
    troubleshooting_agent = Agent(
        role="Troubleshooting Specialist",
        goal="Produce step-by-step user-facing troubleshooting instructions, assign severity levels, and estimate resolution probability.",
        backstory="Experienced in writing clear, actionable support steps that users can follow independently.",
        llm=None
    )
    
    escalation_agent = Agent(
        role="Engineering Escalation Manager",
        goal="Decide whether to escalate issues based on confidence, severity, and product rules; format engineering ticket payloads.",
        backstory="Manages escalation workflows, prioritizes engineering workload, and ensures critical issues reach the right teams.",
        llm=None
    )
    
    def run_pipeline_crewai(user_query: str, product_rules: dict = {}) -> dict:
        """Run the SaaS support workflow using CrewAI orchestration."""
        if not can_use_crewai(True):
            raise RuntimeError('CrewAI is not available.')
        
        task_diagnosis = Task(
            description=f"Analyze the user query and produce a diagnosis. Query: {user_query}",
            agent=tech_diagnosis_agent,
            expected_output="JSON with issue_type, probable_cause, confidence (0-1), actions array, and raw evidence."
        )
        
        task_product_rules = Task(
            description=f"Apply product rules to validate diagnosis. Product rules: {json.dumps(product_rules)}",
            agent=product_knowledge_agent,
            expected_output="Updated state JSON with notes array and escalation flags as needed."
        )
        
        task_troubleshooting = Task(
            description="Generate step-by-step troubleshooting response with severity level.",
            agent=troubleshooting_agent,
            expected_output="JSON with response object containing steps array and severity field."
        )
        
        task_escalation = Task(
            description="Determine if escalation is needed and format the engineering ticket.",
            agent=escalation_agent,
            expected_output="JSON with escalation object containing bool escalate flag, reasons array, and ticket details."
        )
        
        crew = Crew(
            agents=[tech_diagnosis_agent, product_knowledge_agent, troubleshooting_agent, escalation_agent],
            tasks=[task_diagnosis, task_product_rules, task_troubleshooting, task_escalation],
            verbose=False
        )
        
        crew_output = crew.kickoff()
        
        try:
            result = json.loads(crew_output)
        except Exception:
            result = {'raw_output': str(crew_output), 'escalation': {'escalate': False, 'reasons': []}}
        
        return _normalize_state(result, user_query)


def run_pipeline_auto(user_query: str, product_rules: dict = {}, use_crewai: bool = True) -> dict:
    """Auto-select pipeline: CrewAI > LLM > Deterministic."""
    if can_use_crewai(use_crewai):
        try:
            print('[CrewAI]', end=' ')
            return run_pipeline_crewai(user_query, product_rules)
        except Exception as e:
            print(f'CrewAI failed ({e}), trying LLM...')
    
    if USE_LLM:
        try:
            print('[LLM]', end=' ')
            result = run_pipeline_llm(user_query, product_rules)
            if is_valid_pipeline_result(result):
                return result
            print('LLM output schema mismatch, trying deterministic...')
        except Exception as e:
            print(f'LLM failed ({e}), trying deterministic...')
    
    print('[Deterministic]', end=' ')
    return run_pipeline(user_query, product_rules)

EXAMPLES = [
    'I cannot login to my account since yesterday, SSO fails with 401',
    'Our API returns 500 when calling /v1/data with this payload',
    'How do I enable feature X? The docs are unclear',
    'Intermittent data loss after bulk import - production issue',
    'When uploading CSV, some rows disappear and we lost 200 records',
    'The integration with partner app times out after 30s and causes duplicate entries',
    'Dev token is rejected with 403 even though permissions were set',
    'Random 502 from CDN causing feature toggle failures in prod'
]

print("\n" + "="*80)
print("Running SaaS Support Pipeline (CrewAI > LLM > Deterministic):")
print("="*80 + "\n")
for q in EXAMPLES:
    print('---')
    print('Query:', q)
    out = run_pipeline_auto(q, {'sso_outage': False, 'rate_limit_policy': True}, use_crewai=True)
    print('Diagnosis:', out.get('diagnosis'))
    print('Confidence:', out.get('confidence'))
    print('Response steps:', out.get('response', {}).get('steps'))
    print('Escalation:', out.get('escalation'))
    print()

Running deterministic pipeline (no LLM or CrewAI):

---
Query: I cannot login to my account since yesterday, SSO fails with 401
Diagnosis: {'issue_type': 'auth', 'probable_cause': 'invalid_credentials_or_sso'}
Confidence: 0.8
Response steps: ['Confirm username/email is correct', 'Reset password or try SSO login flow', 'If using SSO, check IdP status']
Escalation: {'escalate': False, 'reasons': []}

---
Query: Our API returns 500 when calling /v1/data with this payload
Diagnosis: {'issue_type': 'api', 'probable_cause': 'bad_request_or_backend_error'}
Confidence: 0.6
Response steps: ['Collect request/response headers and payload', 'Check service status and recent deploys', 'Retry the request and capture logs']
Escalation: {'escalate': True, 'reasons': ['product_rule'], 'ticket': {'title': 'Escalation: api', 'description': "our api returns 500 when calling /v1/data with this payload\n{'issue_type': 'api', 'probable_cause': 'bad_request_or_backend_error'}", 'attachments': []}}

---
Query: 

In [ ]:
# Workaround for Chroma/sqlite3 compatibility issues in Colab/Vanilla environments
# Run this cell before importing any Chroma-related packages.

import sys
import subprocess

try:
    import sqlite3
    print("Current sqlite3 version:", sqlite3.sqlite_version)
except Exception as e:
    print("sqlite3 import check failed:", e)

try:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "pysqlite3-binary"])
except Exception as e:
    print("pip install failed:", e)

try:
    import pysqlite3.dbapi2 as sqlite3_mod
    sys.modules["sqlite3"] = sqlite3_mod
    import sqlite3
    print("Patched sqlite3 version:", sqlite3.sqlite_version)
except Exception as e:
    print("SQLite patch failed:", e)
